# MediSight AI — Clinical Risk Prediction & Explainable AI

This notebook documents the current ML pipeline used by MediSight AI. It uses Synthea-generated synthetic healthcare records to build a forward-looking clinical risk model and explain predictions with SHAP.

> **Important:** Synthea data is synthetic. These results demonstrate the MVP ML pipeline and do not represent clinical performance or medical advice.

## 1. ML objective

Predict whether a patient will have a future inpatient or emergency encounter using information available up to a patient-specific cutoff date. The model is intended as **decision support**, not diagnosis.

## 2. Dataset

The current MVP uses **30,921 Synthea-generated synthetic patient records** from two populations. The feature table was created by `ml/src/build_synthea_features.py` from patient, encounter, condition, medication, and relevant observation/lab data.

The target is a forward-looking inpatient/emergency encounter label. Because the source is synthetic, the model metrics should be interpreted as engineering/ML validation results rather than clinical evidence.

In [2]:

%pip install duckdb pandas scikit-learn xgboost joblib


   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
    --------------------------------------- 0.3/13.2 MB ? eta -:--:--
   --- ------------------------------------ 1.3/13.2 MB 5.2 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/13.2 MB 5.2 MB/s eta 0:00:03
   ---------- ----------------------------- 3.4/13.2 MB 5.3 MB/s eta 0:00:02
   -------------- ------------------------- 4.7/13.2 MB 5.7 MB/s eta 0:00:02
   --------------- ------------------------ 5.2/13.2 MB 5.4 MB/s eta 0:00:02
   ------------------ --------------------- 6.0/13.2 MB 4.7 MB/s eta 0:00:02
   ----------------------- ---------------- 7.6/13.2 MB 5.0 MB/s eta 0:00:02
   ------------------------- -------------- 8.4/13.2 MB 5.0 MB/s eta 0:00:01
   -------------------------- ------------- 8.7/13.2 MB 4.9 MB/s eta 0:00:01
   --------------------------- ------------ 9.2/13.2 MB 4.2 MB/s eta 0:00:01
   -----------------

In [3]:

import sys
import duckdb
import pandas as pd

print("Python:", sys.executable)
print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)


Python: c:\Users\ASUS\OneDrive\Desktop\HACKMATRIX 5.0\medisight-ai-updated\medisight-ai\.venv\Scripts\python.exe
DuckDB: 1.5.6
Pandas: 3.0.6


In [5]:

from pathlib import Path
import duckdb
import pandas as pd

project = Path.cwd()

pop1 = Path(
    r"C:\Users\ASUS\OneDrive\Desktop\HACKMATRIX 5.0\dataverse_files\synthea-patient-pop1-csv"
)
pop2 = Path(
    r"C:\Users\ASUS\OneDrive\Desktop\HACKMATRIX 5.0\dataverse_files\synthea-patient-pop2-csv"
)

print("Notebook folder:", project)
print("DuckDB version:", duckdb.__version__)
print("Population 1 exists:", pop1.exists())
print("Population 2 exists:", pop2.exists())
print("Patients CSV 1 exists:", (pop1 / "patients.csv").exists())
print("Patients CSV 2 exists:", (pop2 / "patients.csv").exists())


Notebook folder: c:\Users\ASUS\OneDrive\Desktop\HACKMATRIX 5.0\medisight-ai-updated\medisight-ai\ml\notebooks
DuckDB version: 1.5.6
Population 1 exists: True
Population 2 exists: True
Patients CSV 1 exists: True
Patients CSV 2 exists: True


In [6]:

import duckdb

for label, folder in [
    ("Population 1", pop1),
    ("Population 2", pop2),
]:
    con = duckdb.connect()
    count = con.execute(
        "SELECT COUNT(*) FROM read_csv_auto(?)",
        [str(folder / "patients.csv")]
    ).fetchone()[0]
    con.close()
    print(f"{label}: {count:,} patients")


Population 1: 15,354 patients
Population 2: 15,591 patients


In [7]:

from pathlib import Path

project_root = Path.cwd().parents[1]

print("Project root:", project_root)
print("Feature builder exists:", (project_root / "ml/src/build_synthea_features.py").exists())
print("Backend feature schema exists:", (project_root / "backend/app/ml/features.py").exists())
print("ML data folder exists:", (project_root / "ml/data").exists())


Project root: c:\Users\ASUS\OneDrive\Desktop\HACKMATRIX 5.0\medisight-ai-updated\medisight-ai
Feature builder exists: True
Backend feature schema exists: True
ML data folder exists: True


In [8]:

from pathlib import Path

schema_file = project_root / "backend" / "app" / "ml" / "features.py"
builder_file = project_root / "ml" / "src" / "build_synthea_features.py"

print("=== BACKEND FEATURE SCHEMA ===")
print(schema_file.read_text(encoding="utf-8"))

print("\n=== FEATURE BUILDER: FIRST 100 LINES ===")
lines = builder_file.read_text(encoding="utf-8").splitlines()
for number, line in enumerate(lines[:100], start=1):
    print(f"{number:03}: {line}")


=== BACKEND FEATURE SCHEMA ===
"""
Single source of truth for feature engineering.

The training script (ml/src/train_model.py) and the live inference service
(app/services/risk_service.py) both import FEATURE_NAMES and
build_feature_row from here, so what the model was trained on is exactly
what it sees at prediction time.
"""

FEATURE_NAMES = [
    "age",
    "is_male",
    "num_active_conditions",
    "num_visits_last_year",
    "num_prescriptions",
    "abnormal_lab_ratio",
    "has_diabetes",
    "has_hypertension",
    "has_heart_disease",
    "latest_glucose",
    "latest_systolic_bp",
]


def build_feature_row(record: dict) -> dict:
    """
    `record` is a plain dict describing one patient's aggregated clinical
    picture (see risk_service.summarize_patient_for_model for the shape).
    Returns a dict keyed exactly by FEATURE_NAMES, in a form XGBoost can
    consume (all numeric).
    """
    conditions = {c.lower() for c in record.get("condition_names", [])}

    return {
 

In [9]:

# Check existing builder functions and configuration without printing the whole file
import ast

source = builder_file.read_text(encoding="utf-8")
tree = ast.parse(source)

print("=== BUILDER FUNCTIONS ===")
for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(f"Line {node.lineno}: {node.name}()")

print("\n=== CONFIGURATION / OUTPUT REFERENCES ===")
for number, line in enumerate(source.splitlines(), start=1):
    if any(word in line.upper() for word in [
        "POPULATIONS", "LAB_CODES", "OUTPUT_PATH",
        "to_csv", "def main", "if __name__"
    ]):
        print(f"{number:03}: {line[:180]}")

print("\n=== EXISTING OUTPUT / MODEL FILES ===")
for relative in [
    "ml/data/synthea_features.csv",
    "ml/models/risk_model.joblib",
    "ml/models/model_metadata.json",
]:
    path = project_root / relative
    print(f"{relative}: exists={path.exists()}")
    if path.exists():
        print(f"  size={path.stat().st_size:,} bytes")


=== BUILDER FUNCTIONS ===
Line 52: read_csv_from_zip()
Line 57: process_population()
Line 87: get_cutoff_date()
Line 179: is_abnormal()

=== CONFIGURATION / OUTPUT REFERENCES ===
002: Builds a real-patient feature table from the two Synthea CSV populations
022: POPULATIONS = {
027: LAB_CODES = {
159:         sub = chunk[chunk["CODE"].isin(LAB_CODES.keys())].copy()
171:     obs_past["lab_name"] = obs_past["CODE"].map(LAB_CODES)
232:     for name, path in POPULATIONS.items():
235:     OUTPUT_PATH = os.path.join(
242: full.to_csv(OUTPUT_PATH, index=False)

=== EXISTING OUTPUT / MODEL FILES ===
ml/data/synthea_features.csv: exists=True
  size=3,524,865 bytes
ml/models/risk_model.joblib: exists=True
  size=245,095 bytes
ml/models/model_metadata.json: exists=True
  size=781 bytes


In [10]:

import pandas as pd

features_path = project_root / "ml" / "data" / "synthea_features.csv"
df = pd.read_csv(features_path)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values per column:")
display(df.isna().sum())

print("\nLabel distribution:")
if "label" in df.columns:
    display(df["label"].value_counts(dropna=False))
else:
    print("No column named 'label' found.")


Dataset shape: (30921, 14)

Column names:
['PATIENT', 'age', 'is_male', 'num_active_conditions', 'num_visits_last_year', 'num_prescriptions', 'abnormal_lab_ratio', 'has_diabetes', 'has_hypertension', 'has_heart_disease', 'latest_glucose', 'latest_systolic_bp', 'label', 'population']

First 5 rows:


,PATIENT,age,is_male,num_active_conditions,num_visits_last_year,num_prescriptions,abnormal_lab_ratio,has_diabetes,has_hypertension,has_heart_disease,latest_glucose,latest_systolic_bp,label,population
0,5605b66b-e92d-c16c-1b83-b8bf7040d51f,41.478439,0.0,9.0,2,4.0,0.875000,0.0,1.0,0.0,90.0,147.0,1,pop1
1,c06a2ab3-c3b1-a3d0-7c12-262a12c6885e,75.397673,1.0,8.0,2,4.0,0.000000,0.0,0.0,1.0,90.0,120.0,0,pop1
2,6e5ae27c-8038-7988-e2c0-25a103f01bfa,68.936345,1.0,6.0,1,2.0,0.000000,0.0,1.0,0.0,90.0,120.0,1,pop1
3,94d39619-8e17-d248-41b3-9ec1f2d031b2,53.221081,1.0,9.0,14,2.0,0.301887,0.0,0.0,0.0,89.7,118.0,1,pop1
4,23832f5d-e045-2541-1626-a65dce9bbcf7,42.529774,0.0,6.0,1,4.0,0.142857,0.0,0.0,1.0,90.0,117.0,0,pop1



Missing values per column:


PATIENT                  0
age                      0
is_male                  0
num_active_conditions    0
num_visits_last_year     0
num_prescriptions        0
abnormal_lab_ratio       0
has_diabetes             0
has_hypertension         0
has_heart_disease        0
latest_glucose           0
latest_systolic_bp       0
label                    0
population               0
dtype: int64


Label distribution:


label
1    20188
0    10733
Name: count, dtype: int64

In [11]:

from pathlib import Path

expected_features = [
    "age",
    "is_male",
    "num_active_conditions",
    "num_visits_last_year",
    "num_prescriptions",
    "abnormal_lab_ratio",
    "has_diabetes",
    "has_hypertension",
    "has_heart_disease",
    "latest_glucose",
    "latest_systolic_bp",
]

print("Expected features present:",
      all(c in df.columns for c in expected_features))

print("\nPopulation counts:")
display(df["population"].value_counts(dropna=False))

print("\nFeature summary:")
display(df[expected_features].describe().T)

print("\nLabel rate by population:")
display(df.groupby("population")["label"].agg(["count", "mean"]))

print("\nDuplicate patient rows:", df["PATIENT"].duplicated().sum())


Expected features present: True

Population counts:


population
pop2    15577
pop1    15344
Name: count, dtype: int64


Feature summary:


,count,mean,std,min,25%,50%,75%,max
age,30921.0,59.395386,19.172741,1.007529,44.974675,61.245722,75.033539,97.127995
is_male,30921.0,0.534653,0.498806,0.000000,0.000000,1.000000,1.000000,1.000000
num_active_conditions,30921.0,10.282203,4.064763,0.000000,8.000000,10.000000,12.000000,32.000000
num_visits_last_year,30921.0,7.888975,16.551570,1.000000,2.000000,3.000000,8.000000,240.000000
num_prescriptions,30921.0,3.126096,2.603581,0.000000,1.000000,3.000000,5.000000,21.000000
abnormal_lab_ratio,30921.0,0.430622,0.231812,0.000000,0.315789,0.444444,0.555556,1.000000
has_diabetes,30921.0,0.479803,0.499600,0.000000,0.000000,0.000000,1.000000,1.000000
has_hypertension,30921.0,0.320624,0.466724,0.000000,0.000000,0.000000,1.000000,1.000000
has_heart_disease,30921.0,0.168170,0.374024,0.000000,0.000000,0.000000,0.000000,1.000000
latest_glucose,30921.0,87.573390,11.399369,64.000000,83.100000,90.000000,90.000000,199.300000



Label rate by population:


,count,mean
population,,
pop1,15344,0.650417
pop2,15577,0.655325



Duplicate patient rows: 0


In [12]:

print("=== LABEL DISTRIBUTION BY POPULATION ===")
display(
    df.groupby("population")["label"]
      .agg(["count", "sum", "mean"])
      .rename(columns={"sum": "positive_labels", "mean": "positive_rate"})
)

print("\n=== DUPLICATE PATIENT IDs ===")
print("Duplicate PATIENT rows:", df["PATIENT"].duplicated().sum())

print("\n=== LABEL VALUES ===")
print(df["label"].value_counts(dropna=False).sort_index())

print("\n=== MISSING EXPECTED FEATURES ===")
expected_features = [
    "age", "is_male", "num_active_conditions",
    "num_visits_last_year", "num_prescriptions",
    "abnormal_lab_ratio", "has_diabetes",
    "has_hypertension", "has_heart_disease",
    "latest_glucose", "latest_systolic_bp"
]
print("Missing feature columns:",
      [c for c in expected_features if c not in df.columns])

print("\n=== POPULATION VALUES ===")
print(sorted(df["population"].dropna().unique().tolist()))


=== LABEL DISTRIBUTION BY POPULATION ===


,count,positive_labels,positive_rate
population,,,
pop1,15344,9980,0.650417
pop2,15577,10208,0.655325



=== DUPLICATE PATIENT IDs ===
Duplicate PATIENT rows: 0

=== LABEL VALUES ===
label
0    10733
1    20188
Name: count, dtype: int64

=== MISSING EXPECTED FEATURES ===
Missing feature columns: []

=== POPULATION VALUES ===
['pop1', 'pop2']


In [13]:

import json

metadata_path = project_root / "ml" / "models" / "model_metadata.json"

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2))


{
  "version": "xgb-synthea-v1-20260923114430",
  "trained_at": "2026-09-23T11:44:30.476803+00:00",
  "feature_names": [
    "age",
    "is_male",
    "num_active_conditions",
    "num_visits_last_year",
    "num_prescriptions",
    "abnormal_lab_ratio",
    "has_diabetes",
    "has_hypertension",
    "has_heart_disease",
    "latest_glucose",
    "latest_systolic_bp"
  ],
  "metrics": {
    "accuracy": 0.6768,
    "precision": 0.7921,
    "recall": 0.6846,
    "f1": 0.7344,
    "roc_auc": 0.7389,
    "n_train": 23190,
    "n_test": 7731
  },
  "dataset": "synthea-30921 (pop1 + pop2, Synthea-generated synthetic patients; label = real Synthea encounter history, future inpatient/ER utilization, not a fabricated heuristic)",
  "n_samples": 30921
}


In [14]:

import joblib

model_path = project_root / "ml" / "models" / "risk_model.joblib"

model = joblib.load(model_path)

print("Model loaded:", type(model).__name__)
print("Expected feature count:", model.n_features_in_)
print("Dataset feature count:", len(expected_features))
print("Feature count matches:", model.n_features_in_ == len(expected_features))
print("Metadata feature order matches backend:",
      metadata["feature_names"] == expected_features)


Model loaded: XGBClassifier
Expected feature count: 11
Dataset feature count: 11
Feature count matches: True
Metadata feature order matches backend: True


In [15]:

import sys

backend_path = str(project_root)
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

from backend.app.ml.features import FEATURE_NAMES, build_feature_row

sample_record = {
    "age": 55,
    "is_male": 1,
    "num_active_conditions": 3,
    "num_visits_last_year": 4,
    "num_prescriptions": 2,
    "abnormal_lab_ratio": 0.25,
    "has_diabetes": 1,
    "has_hypertension": 1,
    "has_heart_disease": 0,
    "latest_glucose": 110,
    "latest_systolic_bp": 135,
}

print("Feature names match:", FEATURE_NAMES == metadata["feature_names"])

print("\nBackend feature builder:")
print(build_feature_row(sample_record))


Feature names match: True

Backend feature builder:
{'age': 55.0, 'is_male': 0.0, 'num_active_conditions': 3.0, 'num_visits_last_year': 4.0, 'num_prescriptions': 2.0, 'abnormal_lab_ratio': 0.25, 'has_diabetes': 0.0, 'has_hypertension': 0.0, 'has_heart_disease': 0.0, 'latest_glucose': 110.0, 'latest_systolic_bp': 135.0}


In [16]:

import inspect
print(inspect.getsource(build_feature_row))


def build_feature_row(record: dict) -> dict:
    """
    `record` is a plain dict describing one patient's aggregated clinical
    picture (see risk_service.summarize_patient_for_model for the shape).
    Returns a dict keyed exactly by FEATURE_NAMES, in a form XGBoost can
    consume (all numeric).
    """
    conditions = {c.lower() for c in record.get("condition_names", [])}

    return {
        "age": float(record.get("age", 0)),
        "is_male": 1.0 if str(record.get("gender", "")).lower().startswith("m") else 0.0,
        "num_active_conditions": float(record.get("num_active_conditions", 0)),
        "num_visits_last_year": float(record.get("num_visits_last_year", 0)),
        "num_prescriptions": float(record.get("num_prescriptions", 0)),
        "abnormal_lab_ratio": float(record.get("abnormal_lab_ratio", 0.0)),
        "has_diabetes": 1.0 if any("diabetes" in c for c in conditions) else 0.0,
        "has_hypertension": 1.0 if any("hypertension" in c for c in conditions) els

In [17]:

sample_record = {
    "age": 55,
    "gender": "M",
    "num_active_conditions": 3,
    "num_visits_last_year": 4,
    "num_prescriptions": 2,
    "abnormal_lab_ratio": 0.25,
    "condition_names": [
        "Type 2 diabetes",
        "Essential hypertension",
    ],
    "latest_glucose": 110,
    "latest_systolic_bp": 135,
}

feature_row = build_feature_row(sample_record)

print("Feature row:")
print(feature_row)

print("\nFeature order matches:",
      list(feature_row.keys()) == FEATURE_NAMES)

print("\nDiabetes flag:", feature_row["has_diabetes"])
print("Hypertension flag:", feature_row["has_hypertension"])
print("Male flag:", feature_row["is_male"])


Feature row:
{'age': 55.0, 'is_male': 1.0, 'num_active_conditions': 3.0, 'num_visits_last_year': 4.0, 'num_prescriptions': 2.0, 'abnormal_lab_ratio': 0.25, 'has_diabetes': 1.0, 'has_hypertension': 1.0, 'has_heart_disease': 0.0, 'latest_glucose': 110.0, 'latest_systolic_bp': 135.0}

Feature order matches: True

Diabetes flag: 1.0
Hypertension flag: 1.0
Male flag: 1.0


In [18]:

import pandas as pd

X_sample = pd.DataFrame(
    [[feature_row[name] for name in FEATURE_NAMES]],
    columns=FEATURE_NAMES,
)

prediction = int(model.predict(X_sample)[0])
probabilities = model.predict_proba(X_sample)[0]

print("Prediction class:", prediction)
print("Class probabilities:", model.classes_)
print("Probabilities:", probabilities.round(4))
print("Probability sum:", round(float(probabilities.sum()), 6))


Prediction class: 0
Class probabilities: [0 1]
Probabilities: [0.6738 0.3262]
Probability sum: 1.0


In [19]:

from pathlib import Path

builder_path = project_root / "ml" / "src" / "build_synthea_features.py"
trainer_path = project_root / "ml" / "src" / "train_model_synthea.py"

print("=== FEATURE BUILDER: LABEL / OUTPUT LOGIC ===")
builder_lines = builder_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(builder_lines, start=1):
    if any(term in line.lower() for term in [
        "label", "future", "inpatient", "emergency",
        "to_csv", "output_path", "process_population"
    ]):
        start = max(1, i - 3)
        end = min(len(builder_lines), i + 5)
        print(f"\n--- Lines {start}-{end} ---")
        for j in range(start, end + 1):
            print(f"{j}: {builder_lines[j-1]}")

print("\n=== TRAINING SCRIPT ===")
if trainer_path.exists():
    print(trainer_path.read_text(encoding="utf-8"))
else:
    print("Training script not found:", trainer_path)


=== FEATURE BUILDER: LABEL / OUTPUT LOGIC ===

--- Lines 4-12 ---
4: used by app/ml/features.py (FEATURE_NAMES) — so the resulting CSV can be
5: dropped straight into training in place of the 100-patient Faker dev seed.
6: 
7: Label (real, not heuristic):
8:   For each patient, pick a cutoff point at ~70% of their encounter history.
9:   Features are computed from everything up to that cutoff ("current picture").
10:   Label = 1 if the patient has an INPATIENT or EMERGENCY encounter strictly
11:   AFTER the cutoff, else 0. This is a genuine forward-looking prediction
12:   target (future acute-care utilization), not a fabricated score+noise.

--- Lines 7-15 ---
7: Label (real, not heuristic):
8:   For each patient, pick a cutoff point at ~70% of their encounter history.
9:   Features are computed from everything up to that cutoff ("current picture").
10:   Label = 1 if the patient has an INPATIENT or EMERGENCY encounter strictly
11:   AFTER the cutoff, else 0. This is a genuine forward

In [20]:

trainer_path = project_root / "ml" / "src" / "train_model_synthea.py"

print(trainer_path.read_text(encoding="utf-8"))


"""
Train the MediSight risk model on REAL Synthea data (30,921 patients across
pop1 + pop2), instead of the 100-patient Faker dev seed.

Drop this file into ml/src/ alongside the existing train_model.py, and put
synthea_features.csv into ml/data/.

WHAT CHANGED vs the original train_model.py
---------------------------------------------
- Data source: reads ml/data/synthea_features.csv (pre-extracted from the
  two Synthea CSV exports) instead of querying the Postgres `patients` table
  seeded by scripts/seed_data.py (Faker, n=100).
- Label: no more heuristic-score-plus-noise. The label here is a REAL
  observed outcome from the Synthea simulation: for each patient we cut
  their encounter history at ~70% of the way through, compute features from
  everything up to that point, and label = 1 if that patient went on to
  have an INPATIENT or EMERGENCY encounter afterwards. That's a genuine
  forward-looking prediction target, not a fabricated score.
- Everything else (feature set, XGBoo

In [21]:

builder_path = project_root / "ml" / "src" / "build_synthea_features.py"
lines = builder_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if line.startswith("def main(") or line.startswith("if __name__"):
        start = max(0, i - 3)
        end = min(len(lines), i + 70)
        print("\n".join(
            f"{j+1}: {lines[j]}" for j in range(start, end)
        ))


227:     return out
228: 
229: 
230: if __name__ == "__main__":
231:     all_frames = []
232:     for name, path in POPULATIONS.items():
233:         all_frames.append(process_population(name, path))
234:     full = pd.concat(all_frames, ignore_index=True)
235:     OUTPUT_PATH = os.path.join(
236:     os.path.dirname(__file__),
237:     "..",
238:     "data",
239:     "synthea_features.csv"
240: )
241: 
242: full.to_csv(OUTPUT_PATH, index=False)
243: print(
244:     f"\nTOTAL: {len(full)} patients, " 
245:     f"{full['label'].sum()} positive "
246:     f"({100*full['label'].mean():.1f}%)"
247: )


In [22]:

trainer_path = project_root / "ml" / "src" / "train_model_synthea.py"

print("Trainer exists:", trainer_path.exists())

if trainer_path.exists():
    trainer_text = trainer_path.read_text(encoding="utf-8")
    print("Total lines:", len(trainer_text.splitlines()))

    for start in range(0, len(trainer_text.splitlines()), 60):
        end = min(start + 60, len(trainer_text.splitlines()))
        print(f"\n--- Lines {start + 1}-{end} ---")
        print("\n".join(trainer_text.splitlines()[start:end]))


Trainer exists: True
Total lines: 131

--- Lines 1-60 ---
"""
Train the MediSight risk model on REAL Synthea data (30,921 patients across
pop1 + pop2), instead of the 100-patient Faker dev seed.

Drop this file into ml/src/ alongside the existing train_model.py, and put
synthea_features.csv into ml/data/.

WHAT CHANGED vs the original train_model.py
---------------------------------------------
- Data source: reads ml/data/synthea_features.csv (pre-extracted from the
  two Synthea CSV exports) instead of querying the Postgres `patients` table
  seeded by scripts/seed_data.py (Faker, n=100).
- Label: no more heuristic-score-plus-noise. The label here is a REAL
  observed outcome from the Synthea simulation: for each patient we cut
  their encounter history at ~70% of the way through, compute features from
  everything up to that point, and label = 1 if that patient went on to
  have an INPATIENT or EMERGENCY encounter afterwards. That's a genuine
  forward-looking prediction target, not

In [23]:

trainer_lines = trainer_path.read_text(encoding="utf-8").splitlines()

print("\n".join(
    f"{i+1}: {trainer_lines[i]}"
    for i in range(60, len(trainer_lines))
))


61:         )
62: 
63:     df = pd.read_csv(DATA_PATH)
64:     print(f"Loaded {len(df)} real Synthea patients "
65:           f"({df['population'].nunique()} populations)")
66:     print(f"Label balance: {df['label'].sum()} positive / "
67:           f"{len(df) - df['label'].sum()} negative "
68:           f"({100*df['label'].mean():.1f}% positive)")
69: 
70:     X = df[FEATURE_NAMES]
71:     y = df["label"]
72: 
73:     X_train, X_test, y_train, y_test = train_test_split(
74:         X, y, test_size=0.25, random_state=42, stratify=y
75:     )
76: 
77:     n_pos = max(int(y_train.sum()), 1)
78:     n_neg = max(len(y_train) - n_pos, 1)
79:     scale_pos_weight = n_neg / n_pos
80: 
81:     model = XGBClassifier(
82:         n_estimators=150,
83:         max_depth=4,
84:         learning_rate=0.08,
85:         subsample=0.9,
86:         colsample_bytree=0.9,
87:         eval_metric="logloss",
88:         scale_pos_weight=scale_pos_weight,
89:         random_state=42,
90:     )
91:     mod

In [24]:

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

data_path = project_root / "ml" / "data" / "synthea_features.csv"
model_path = project_root / "ml" / "models" / "risk_model.joblib"
metadata_path = project_root / "ml" / "models" / "model_metadata.json"

# Validate dataset before training
df_check = pd.read_csv(data_path)

assert "label" in df_check.columns, "Missing label column"
assert all(c in df_check.columns for c in FEATURE_NAMES), "Missing model features"
assert df_check["label"].isin([0, 1]).all(), "Labels must be 0 or 1"
assert not df_check[FEATURE_NAMES].isna().any().any(), "Missing feature values"

print("Dataset shape:", df_check.shape)
print("Unique patients:", df_check["PATIENT"].nunique())
print("Populations:", df_check["population"].value_counts().to_dict())
print("Label counts:", df_check["label"].value_counts().sort_index().to_dict())

# Timestamped backup of current model artifacts
backup_dir = project_root / "ml" / "models" / (
    "backup_before_synthea_retrain_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
backup_dir.mkdir(parents=True, exist_ok=False)

for p in [model_path, metadata_path]:
    if p.exists():
        shutil.copy2(p, backup_dir / p.name)

print("\nBackup folder:", backup_dir)
print("Backup files:", [p.name for p in backup_dir.iterdir()])


Dataset shape: (30921, 14)
Unique patients: 30921
Populations: {'pop2': 15577, 'pop1': 15344}
Label counts: {0: 10733, 1: 20188}

Backup folder: c:\Users\ASUS\OneDrive\Desktop\HACKMATRIX 5.0\medisight-ai-updated\medisight-ai\ml\models\backup_before_synthea_retrain_20261010_132712
Backup files: ['model_metadata.json', 'risk_model.joblib']


In [25]:

import json
import joblib
import pandas as pd

model_path = project_root / "ml" / "models" / "risk_model.joblib"
metadata_path = project_root / "ml" / "models" / "model_metadata.json"

# Load the newly saved model and metadata
trained_model = joblib.load(model_path)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

from backend.app.ml.features import FEATURE_NAMES, build_feature_row

sample_record = {
    "age": 55,
    "gender": "M",
    "num_active_conditions": 3,
    "num_visits_last_year": 4,
    "num_prescriptions": 2,
    "abnormal_lab_ratio": 0.25,
    "condition_names": ["Type 2 diabetes", "Essential hypertension"],
    "latest_glucose": 110,
    "latest_systolic_bp": 135,
}

row = build_feature_row(sample_record)
X_sample = pd.DataFrame(
    [[row[name] for name in FEATURE_NAMES]],
    columns=FEATURE_NAMES,
)

assert trained_model.n_features_in_ == len(FEATURE_NAMES)
assert metadata["feature_names"] == FEATURE_NAMES

prediction = int(trained_model.predict(X_sample)[0])
probabilities = trained_model.predict_proba(X_sample)[0]

print("Model loaded:", type(trained_model).__name__)
print("Model features:", trained_model.n_features_in_)
print("Feature order matches backend: PASS")
print("Prediction:", prediction)
print("Classes:", trained_model.classes_)
print("Probabilities:", probabilities.round(4))
print("Probability sum:", round(float(probabilities.sum()), 6))
print("Metadata metrics:", metadata.get("metrics"))


Model loaded: XGBClassifier
Model features: 11
Feature order matches backend: PASS
Prediction: 0
Classes: [0 1]
Probabilities: [0.6738 0.3262]
Probability sum: 1.0
Metadata metrics: {'accuracy': 0.6768, 'precision': 0.7921, 'recall': 0.6846, 'f1': 0.7344, 'roc_auc': 0.7389, 'n_train': 23190, 'n_test': 7731}
